In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_4.py ──
"""
# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 4: Shared Utilities
# ════════════════════════════════════════════════════════════════════════
#
# Common infrastructure for all Exercise 4 technique files:
#   - AG News loading + caching (120K train, 7.6K test)
#   - Vocabulary building and text-to-index conversion
#   - ExperimentTracker + ModelRegistry + OnnxBridge setup
#   - Shared constants, device detection, visualisation helpers
#
# ════════════════════════════════════════════════════════════════════════
"""

import asyncio
import math
from collections import Counter
from pathlib import Path
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset

import plotly.graph_objects as go

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml import OnnxBridge

# ════════════════════════════════════════════════════════════════════════
# Environment + Device
# ════════════════════════════════════════════════════════════════════════
setup_environment()

torch.manual_seed(42)
np.random.seed(42)
DEVICE = get_device()

# ════════════════════════════════════════════════════════════════════════
# Constants
# ════════════════════════════════════════════════════════════════════════
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
MAX_LEN = 40
VAL_FRACTION = 0.05  # share of the AG News TRAIN split held out for model selection
VAL_SPLIT_SEED = 42
VOCAB_SIZE = 15000
EPOCHS_SCRATCH = 8
BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LEN = 64
BERT_EPOCHS = 3
BERT_LR = 2e-5
BERT_BATCH_SIZE = 32


# ════════════════════════════════════════════════════════════════════════
# Core Attention Function (shared across 01, 02, 05)
# ════════════════════════════════════════════════════════════════════════
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot-product attention.

    Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V

    Args:
        q: Query tensor of shape (B, L_q, d_k)
        k: Key tensor of shape (B, L_k, d_k)
        v: Value tensor of shape (B, L_k, d_v)
        mask: Optional mask of shape (B, L_q, L_k). Positions with 0 are
              masked out (set to -inf before softmax).

    Returns:
        (output, attention_weights) where output has shape (B, L_q, d_v)
        and attention_weights has shape (B, L_q, L_k).
    """
    d_k = q.size(-1)
    scores = torch.einsum("bqd,bkd->bqk", q, k) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    out = torch.einsum("bqk,bkd->bqd", weights, v)
    return out, weights


# ════════════════════════════════════════════════════════════════════════
# AG News Loading
# ════════════════════════════════════════════════════════════════════════
def load_ag_news_split(split: str, cache_name: str) -> pl.DataFrame:
    """Load AG News split, caching to parquet for subsequent runs."""
    cache = Path.cwd() / "data" / "mlfp05" / cache_name
    cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        return pl.read_parquet(cache)
    print(f"  Downloading AG News {split} from HuggingFace...")
    ds = load_dataset("fancyzhx/ag_news", split=split)
    df = pl.from_pandas(ds.to_pandas())
    df.write_parquet(cache)
    return df


def load_ag_news() -> tuple[pl.DataFrame, pl.DataFrame]:
    """Load full AG News train + test splits with status reporting."""
    print("\n== Loading FULL AG News (120K train + 7.6K test) ==")
    train_df = load_ag_news_split("train", "ag_news_full_train.parquet")
    test_df = load_ag_news_split("test", "ag_news_full_test.parquet")
    print(f"  train rows: {len(train_df):,}   test rows: {len(test_df):,}")
    print(f"  sample headline: {train_df['text'][0][:80]!r}")
    print(f"  class balance (train): {dict(Counter(train_df['label'].to_list()))}")
    return train_df, test_df


# ════════════════════════════════════════════════════════════════════════
# Vocabulary + Tokenisation (for from-scratch models: LSTM + Transformer)
# ════════════════════════════════════════════════════════════════════════
def build_vocab(texts: list[str], max_vocab: int = VOCAB_SIZE) -> dict[str, int]:
    """Build a word-level vocabulary from text list."""
    words: Counter[str] = Counter()
    for t in texts:
        words.update(t.lower().split())
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, _ in words.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab


def text_to_indices(
    text: str, vocab: dict[str, int], max_len: int = MAX_LEN
) -> list[int]:
    """Convert text to padded index sequence."""
    tokens = text.lower().split()[:max_len]
    idxs = [vocab.get(t, 1) for t in tokens]
    return (idxs + [0] * (max_len - len(idxs)))[:max_len]


def prepare_dataloaders(
    train_df: pl.DataFrame,
    test_df: pl.DataFrame,
    vocab: dict[str, int],
    batch_size: int = 128,
) -> tuple[
    DataLoader, DataLoader, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor
]:
    """Tokenise AG News and build DataLoaders for from-scratch models.

    A fixed 5% of the TRAINING split is held out as the validation set:
    train_model() picks its best epoch on it, so the test split is never
    used for model selection and stays an unbiased final measurement
    (see evaluate_accuracy).

    Returns: (train_loader, val_loader, train_t, train_y, test_t, test_y)
    where train_t/train_y exclude the validation rows.
    """
    train_tokens = np.array(
        [text_to_indices(t, vocab, MAX_LEN) for t in train_df["text"].to_list()],
        dtype=np.int64,
    )
    train_labels = np.array(train_df["label"].to_list(), dtype=np.int64)
    test_tokens = np.array(
        [text_to_indices(t, vocab, MAX_LEN) for t in test_df["text"].to_list()],
        dtype=np.int64,
    )
    test_labels = np.array(test_df["label"].to_list(), dtype=np.int64)

    perm = np.random.default_rng(VAL_SPLIT_SEED).permutation(len(train_tokens))
    n_val = int(round(VAL_FRACTION * len(train_tokens)))
    val_idx, fit_idx = perm[:n_val], perm[n_val:]

    train_t = torch.from_numpy(train_tokens[fit_idx]).to(DEVICE)
    train_y = torch.from_numpy(train_labels[fit_idx]).to(DEVICE)
    val_t = torch.from_numpy(train_tokens[val_idx]).to(DEVICE)
    val_y = torch.from_numpy(train_labels[val_idx]).to(DEVICE)
    test_t = torch.from_numpy(test_tokens).to(DEVICE)
    test_y = torch.from_numpy(test_labels).to(DEVICE)

    train_loader = DataLoader(
        TensorDataset(train_t, train_y), batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(TensorDataset(val_t, val_y), batch_size=batch_size)

    return train_loader, val_loader, train_t, train_y, test_t, test_y


# ════════════════════════════════════════════════════════════════════════
# Kailash-ML Engine Setup
# ════════════════════════════════════════════════════════════════════════
async def _setup_engines_async() -> (
    tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]
):
    """Initialise ExperimentTracker (kailash-ml 1.1.1 factory) + ModelRegistry."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_transformers.db"
    registry_db = "sqlite:///mlfp05_transformers_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_transformers", registry, True


def setup_engines() -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
    OnnxBridge,
]:
    """Set up all kailash-ml engines (sync wrapper).

    Returns: (conn, tracker, exp_name, registry, has_registry, bridge)
    """
    conn, tracker, exp_name, registry, has_registry = asyncio.run(
        _setup_engines_async()
    )
    bridge = OnnxBridge()
    return conn, tracker, exp_name, registry, has_registry, bridge


# ════════════════════════════════════════════════════════════════════════
# Training Utilities
# ════════════════════════════════════════════════════════════════════════
async def train_model_async(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = EPOCHS_SCRATCH,
    lr: float = 2e-3,
) -> tuple[list[float], list[float]]:
    """Train a PyTorch classifier and log every epoch to ExperimentTracker.

    Uses the modern ``tracker.track(...)`` async context manager -- bulk
    param logging, per-step metrics, automatic COMPLETED/FAILED state.
    """
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    train_losses: list[float] = []
    val_accs: list[float] = []
    best_acc = 0.0
    best_state: dict | None = None
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    async with tracker.track(experiment=exp_name, run_name=model_name) as run:
        await run.log_params(
            {
                "model_type": model_name,
                "epochs": str(epochs),
                "lr": str(lr),
                "dataset_size": str(len(train_loader.dataset)),
                "batch_size": str(train_loader.batch_size),
                "trainable_params": str(param_count),
            }
        )

        for epoch in range(epochs):
            model.train()
            batch_losses = []
            for xb, yb in train_loader:
                optimizer.zero_grad()
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                batch_losses.append(loss.item())
            scheduler.step()
            epoch_loss = float(np.mean(batch_losses))
            train_losses.append(epoch_loss)

            model.eval()
            with torch.no_grad():
                correct = 0
                total = 0
                for xb, yb in val_loader:
                    preds = model(xb).argmax(dim=-1)
                    correct += int((preds == yb).sum().item())
                    total += int(yb.size(0))
                acc = correct / total
                val_accs.append(acc)

            await run.log_metrics(
                {"train_loss": epoch_loss, "val_accuracy": acc},
                step=epoch + 1,
            )
            if acc > best_acc:
                best_acc = acc
                best_state = {
                    k: v.detach().clone() for k, v in model.state_dict().items()
                }
            print(
                f"  [{model_name}] epoch {epoch+1}/{epochs}  "
                f"loss={epoch_loss:.4f}  val_acc={acc:.3f}"
            )

        await run.log_metrics(
            {
                "best_val_accuracy": best_acc,
                "final_train_loss": train_losses[-1],
            }
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return train_losses, val_accs


def train_model(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = EPOCHS_SCRATCH,
    lr: float = 2e-3,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model, model_name, train_loader, val_loader, tracker, exp_name, epochs, lr
        )
    )


def evaluate_accuracy(
    model: nn.Module, tokens: torch.Tensor, labels: torch.Tensor, batch_size: int = 512
) -> float:
    """Accuracy of `model` on (tokens, labels) — use it on the TEST split once,
    after train_model() has restored the validation-selected checkpoint."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for start in range(0, len(tokens), batch_size):
            preds = model(tokens[start : start + batch_size]).argmax(dim=-1)
            correct += int((preds == labels[start : start + batch_size]).sum().item())
    return correct / len(tokens)


# ════════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ════════════════════════════════════════════════════════════════════════
def create_attention_heatmap(
    attn_weights: np.ndarray,
    labels: list[str],
    title: str = "Self-Attention Heatmap",
    max_tokens: int = 15,
) -> go.Figure:
    """Create a plotly heatmap from an attention weight matrix.

    Args:
        attn_weights: 2D numpy array of shape (seq, seq)
        labels: token labels for axes
        title: chart title
        max_tokens: limit display to first N non-pad tokens

    Returns:
        plotly Figure
    """
    show_len = min(max_tokens, len([w for w in labels if w != "<pad>"]))
    fig = go.Figure(
        data=go.Heatmap(
            z=attn_weights[:show_len, :show_len],
            x=labels[:show_len],
            y=labels[:show_len],
            colorscale="Viridis",
            colorbar={"title": "Attention weight"},
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Key position",
        yaxis_title="Query position",
        width=600,
        height=500,
    )
    return fig


def get_viz() -> ModelVisualizer:
    """Return a ModelVisualizer instance."""
    return ModelVisualizer()




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 4.2: Transformer Encoder Classifier
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Implement multi-head attention with parallel learned projections
#   - Explain how different heads capture different relationship types
#   - Build sinusoidal positional encoding (giving transformers order)
#   - Construct a full Transformer encoder classifier with nn.TransformerEncoder
#   - Train the Transformer on AG News and log metrics with ExperimentTracker
#   - Check whether a trained classifier's labels fit a new task
#     (routing regulatory filings) before using it
#
# PREREQUISITES: ex_4/01_self_attention_from_scratch.py
# ESTIMATED TIME: ~30 min
# DATASET: AG News — 120,000 real news headlines, 4 classes.
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


print(f"Using device: {DEVICE}")



## THEORY — Multi-Head Attention and Positional Encoding


In [ ]:
# A single attention head can only capture one type of relationship at a
# time. But language has many simultaneous relationship types:
#
#   "The bank raised interest rates to control inflation"
#
#   - SYNTACTIC: "bank" -> "raised" (subject-verb)
#   - SEMANTIC: "rates" -> "inflation" (economic cause-effect)
#   - COREFERENCE: "bank" -> "central bank" (if mentioned earlier)
#
# Multi-head attention runs h parallel attention operations, each with
# its own learned Q/K/V projections. Head 1 might learn syntax, head 2
# might learn semantics, head 3 might learn entity relationships. The
# outputs are concatenated and projected back to the model dimension.
#
# POSITIONAL ENCODING: Transformers process all tokens simultaneously
# (unlike RNNs, which process sequentially). This means they have no
# inherent sense of word order. "Dog bites man" and "Man bites dog"
# would look identical without positional information. We inject position
# using a sinusoidal signal:
#   PE(pos, 2i)   = sin(pos / 10000^(2i/d))
#   PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
#
# This scheme gives each position a unique signature and allows the model
# to learn relative positions (the offset between any two positions has
# a consistent geometric relationship in the PE space).



## TASK 1 — Load data and set up engines


In [ ]:
train_df, test_df = load_ag_news()
vocab = build_vocab(train_df["text"].to_list())
train_loader, val_loader, train_t, train_y, test_t, test_y = prepare_dataloaders(
    train_df, test_df, vocab
)
conn, tracker, exp_name, registry, has_registry, bridge = setup_engines()
print(f"  vocab size: {len(vocab)}, seq_len: {MAX_LEN}")
print(f"  ExperimentTracker ready, experiment: {exp_name}")



## TASK 2 — Build: Educational Multi-Head Attention


Each head gets its own learned projection of the input into Q, K, V
    subspaces. The heads operate in parallel, then their outputs are
    concatenated and projected back to the model dimension.


Args:
            x: Input tensor of shape (batch, seq_len, d_model)

        Returns:
            (output, attention_weights) where output has shape
            (batch, seq_len, d_model) and attention_weights has shape
            (batch, n_heads, seq_len, seq_len).


In [ ]:
# This implementation wraps our from-scratch scaled_dot_product_attention
# with learned Q/K/V projections and multi-head splitting. PyTorch has
# nn.MultiheadAttention, but building it ourselves reveals the mechanics.
class EducationalMultiHead(nn.Module):

    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert (
            d_model % n_heads == 0
        ), f"d_model ({d_model}) must be divisible by n_heads ({n_heads})"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # Single linear layer produces Q, K, V for all heads at once
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        b, seq, d = x.shape
        # Compute Q, K, V for all heads in one matrix multiply
        qkv = self.qkv(x).reshape(b, seq, 3, self.n_heads, self.d_k)
        q, k, v = qkv.unbind(dim=2)  # each: (b, seq, n_heads, d_k)

        # Reshape for our attention function: merge batch and head dims
        q = q.transpose(1, 2).reshape(b * self.n_heads, seq, self.d_k)
        k = k.transpose(1, 2).reshape(b * self.n_heads, seq, self.d_k)
        v = v.transpose(1, 2).reshape(b * self.n_heads, seq, self.d_k)

        # Apply our from-scratch attention
        out, weights = scaled_dot_product_attention(q, k, v)

        # Reshape weights to (b, n_heads, seq, seq) for visualisation
        attn_weights = weights.reshape(b, self.n_heads, seq, seq)

        # Concatenate heads and project back
        out = (
            out.reshape(b, self.n_heads, seq, self.d_k)
            .transpose(1, 2)
            .reshape(b, seq, d)
        )
        return self.proj(out), attn_weights


# Sanity check the educational multi-head attention
mha = EducationalMultiHead(d_model=64, n_heads=4).to(DEVICE)
dummy = torch.randn(2, 16, 64, device=DEVICE)
mha_out, mha_attn = mha(dummy)
print(
    f"\nEducationalMultiHead output shape: {tuple(mha_out.shape)}  (expected (2, 16, 64))"
)
print(f"Attention weights shape: {tuple(mha_attn.shape)}  (expected (2, 4, 16, 16))")



### Checkpoint 1


In [ ]:
assert mha_out.shape == (2, 16, 64), "Multi-head output should be (2, 16, 64)"
assert mha_attn.shape == (2, 4, 16, 16), "Attention should be (batch, heads, seq, seq)"
print("\n--- Checkpoint 1 passed --- multi-head attention architecture ready\n")



## TASK 3 — Build: Positional Encoding + Transformer Classifier


Adds a fixed, non-learned signal that encodes position. Each dimension
    oscillates at a different frequency, creating a unique "fingerprint"
    for each position that the model can use to distinguish word order.


Architecture: embedding -> positional encoding -> stacked
    TransformerEncoderLayers -> mean pool -> classification head.

    Uses PyTorch's nn.TransformerEncoder for the stacked layers (which
    internally uses nn.MultiheadAttention), but the architecture mirrors
    our educational implementation above.


In [ ]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class TransformerClassifier(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 3,
        n_classes: int = 4,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.posenc = PositionalEncoding(d_model)
        self.emb_drop = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
        )
        # enable_nested_tensor=False: PyTorch's nested-tensor optimization is
        # not supported on Apple Silicon MPS (raises NotImplementedError on
        # `_nested_tensor_from_mask_left_aligned`). Disabling here ensures the
        # lesson runs on every backend; pedagogical impact is zero.
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=n_layers, enable_nested_tensor=False
        )
        self.head_drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        pad_mask = tokens == 0
        x = self.embed(tokens)
        x = self.posenc(x)
        x = self.emb_drop(x)
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        # Mean-pool over non-pad positions so short headlines aren't diluted
        lengths = (~pad_mask).sum(dim=1, keepdim=True).clamp(min=1).float()
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        pooled = x.sum(dim=1) / lengths
        return self.head(self.head_drop(pooled))



### Checkpoint 2


In [ ]:
tc_test = TransformerClassifier(vocab_size=100).to(DEVICE)
dummy_tokens = torch.randint(0, 100, (2, MAX_LEN), device=DEVICE)
tc_out = tc_test(dummy_tokens)
assert tc_out.shape == (2, 4), "TransformerClassifier should output (batch, 4 classes)"
print("--- Checkpoint 2 passed --- TransformerClassifier architecture ready\n")



## TASK 4 — Train: Transformer on full AG News with ExperimentTracker


In [ ]:
print("\n== Training Transformer on full AG News ==")
transformer_model = TransformerClassifier(
    vocab_size=len(vocab), d_model=128, n_heads=4, n_layers=3, n_classes=4
)
transformer_losses, transformer_accs = train_model(
    transformer_model,
    "transformer",
    train_loader,
    val_loader,
    tracker,
    exp_name,
    epochs=EPOCHS_SCRATCH,
)



## DIAGNOSTIC CHECKPOINT — Transformer (attention + residual stack)


Cross-entropy on one (token_ids, labels) batch.


In [ ]:
# Probes come from the training loader (the probe runs in train mode,
# with dropout active).
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _ce_loss(m, batch):
    xb, yb = batch
    return F.cross_entropy(m(xb), yb)


print("\n── Diagnostic Report (Transformer Encoder) ──")
diag, findings = run_diagnostic_checkpoint(
    transformer_model,
    train_loader,
    _ce_loss,
    title="Transformer Encoder",
    train_losses=transformer_losses,
    show=False,
)
print_prescription_pad(findings, "Transformer Encoder")

# ══════ READING THE PRESCRIPTION PAD (key: see ex_1/01_standard_ae.py) ══════
# nn.TransformerEncoderLayer uses ReLU in its feed-forward block by
# default, so the dead-neuron check applies to those units. Residual
# connections plus LayerNorm give every layer a short gradient path —
# compare the gradient-flow reading with the LSTM in 03. Only training
# loss is passed in; the validation-accuracy curve below shows whether
# more epochs would help.



In [ ]:
# train_model kept the epoch with the best VALIDATION accuracy (a holdout
# carved from the training split); the test split is measured once, here.
transformer_test_acc = evaluate_accuracy(transformer_model, test_t, test_y)



### Checkpoint 3


In [ ]:
assert (
    len(transformer_losses) == EPOCHS_SCRATCH
), "Transformer should train for all epochs"
assert (
    transformer_test_acc > 0.60
), f"Transformer should reach >60% test accuracy, got {transformer_test_acc:.3f}"
# INTERPRETATION: The Transformer processes all tokens in parallel and uses
# self-attention to capture long-range dependencies. On AG News headlines,
# it can directly connect "tech" at position 1 with "stocks" at position 8
# without propagating through every intermediate token. This architectural
# advantage becomes more pronounced on longer documents.
print(f"\n  Transformer: best validation acc {max(transformer_accs):.3f} -> test acc {transformer_test_acc:.3f}")
print("\n--- Checkpoint 3 passed --- Transformer trained on AG News\n")



## TASK 5 — Visualise: Multi-head attention patterns on sample headline


nn.TransformerEncoderLayer (post-norm, the default) feeds its input
    straight into self_attn, so we rebuild that input (embedding +
    positional encoding) and ask the layer's own attention module for its
    weights. Returns (batch, n_heads, seq, seq); padded keys get weight 0.


In [ ]:
print("\n== Visualising multi-head attention patterns ==")
transformer_model.eval()
sample_texts = test_df["text"].to_list()[:3]
sample_idx = torch.tensor(
    [text_to_indices(t, vocab, MAX_LEN) for t in sample_texts],
    dtype=torch.long,
    device=DEVICE,
)

def encoder_attention(model: nn.Module, tokens: torch.Tensor) -> torch.Tensor:
    model.eval()
    pad_mask = tokens == 0
    x = model.posenc(model.embed(tokens))
    first_layer = model.encoder.layers[0]
    _, weights = first_layer.self_attn(
        x,
        x,
        x,
        key_padding_mask=pad_mask,
        need_weights=True,
        average_attn_weights=False,
    )
    return weights


# The heatmaps must come from the layer that was TRAINED: a fresh
# EducationalMultiHead here would show random projections.
with torch.no_grad():
    attn_weights = encoder_attention(transformer_model, sample_idx[:1])  # (1, 4, seq, seq)

words = sample_texts[0].lower().split()[:MAX_LEN]
word_labels = words + ["<pad>"] * (MAX_LEN - len(words))

# Visualise each head's attention pattern
for head_idx in range(min(4, attn_weights.shape[1])):
    attn_np = attn_weights[0, head_idx].cpu().numpy()
    fig = create_attention_heatmap(
        attn_np,
        word_labels,
        title=f"Attention Head {head_idx} on: '{sample_texts[0][:50]}...'",
        max_tokens=12,
    )
    fig.write_html(f"ex_4_2_head_{head_idx}_attention.html")

print(f"  Saved 4 attention head heatmaps (ex_4_2_head_0..3_attention.html)")
print(f"  Compare the four heads: do they spread attention differently?")
print(f"  (Specialisation is something to LOOK FOR, not assume: heads in a")
print(f"  small model trained for a few epochs often look alike.)")



### Checkpoint 4


In [ ]:
assert attn_weights.shape == (
    1,
    4,
    MAX_LEN,
    MAX_LEN,
), "Should have 4 heads of attention"
print("\n--- Checkpoint 4 passed --- multi-head attention visualised\n")



## TASK 6 — Apply: Routing Regulatory Filings (and the Label-Space Trap)


In [ ]:
# SCENARIO: A financial regulator's compliance team receives thousands of
# filings a month and wants each one routed by the regulation it concerns:
#   - Banking Act (Cap. 19)
#   - Securities and Futures Act (Cap. 289)
#   - Payment Services Act 2019
#   - Insurance Act (Cap. 142)
#
# The Transformer you trained answers a different question: its labels
# are AG News topics (World, Sports, Business, Sci/Tech). Run on finance
# headlines it can only say "Business" or similar — never "Insurance
# Act". Routing by regulation needs the same architecture trained on
# filings labelled by Act. What this section CAN show is how to inspect
# which words the trained attention focused on — with the caveat that
# attention weights are a view into the model, not a faithful,
# audit-grade explanation of its decision.
#
# BUSINESS VALUE (illustrative assumptions): at 15-20 minutes of manual
# triage per filing and ~3,000 filings/month, first-pass routing costs
# 750-1,000 officer-hours a month — savings that only exist once a
# classifier is trained on the regulator's own routing labels.
print("\n== Application: routing filings with a news-topic model ==")

# Finance headlines stand in for filings; the model returns NEWS topics.
financial_headlines = [
    "Banks report higher profits amid rising interest rates",
    "New technology startups attract venture capital funding",
    "Stock market volatility increases as trade tensions rise",
    "Sports betting companies face new regulatory scrutiny",
    "Insurance companies adapt to climate change risks",
]

transformer_model.eval()
with torch.no_grad():
    fin_idx = torch.tensor(
        [text_to_indices(t, vocab, MAX_LEN) for t in financial_headlines],
        dtype=torch.long,
        device=DEVICE,
    )
    fin_logits = transformer_model(fin_idx)
    fin_probs = F.softmax(fin_logits, dim=-1)
    fin_preds = fin_logits.argmax(dim=-1).cpu().tolist()

print(f"\n  Finance headlines through the AG News Transformer (topics, not Acts):")
print(f"  {'Headline':<55} {'Topic':<12} {'Confidence':>10}")
print("  " + "-" * 79)
for text, pred, probs in zip(financial_headlines, fin_preds, fin_probs.cpu().tolist()):
    cls_name = CLASS_NAMES[pred]
    confidence = max(probs)
    print(f"  {text[:53]:<55} {cls_name:<12} {confidence:>10.1%}")

# Show attention-based explanation for the first document
with torch.no_grad():
    fin_attn = encoder_attention(transformer_model, fin_idx[:1])  # (1, 4, seq, seq)
    # Average across heads for an aggregate attention view
    avg_attn = fin_attn[0].mean(dim=0).cpu().numpy()  # (seq, seq)

fin_words = financial_headlines[0].lower().split()[:MAX_LEN]
fin_labels = fin_words + ["<pad>"] * (MAX_LEN - len(fin_words))
# Token-level attention: how much total attention each token receives
token_importance = avg_attn[: len(fin_words), : len(fin_words)].sum(axis=0)
token_importance = token_importance / token_importance.max()

print(f"\n  Attention-based explanation for: '{financial_headlines[0]}'")
print(f"  Token importance (which words drive the classification):")
for word, imp in sorted(zip(fin_words, token_importance), key=lambda x: -x[1])[:5]:
    bar = "#" * int(imp * 20)
    print(f"    {word:<15} {imp:.3f} {bar}")



### Checkpoint 5


In [ ]:
assert len(fin_preds) == len(financial_headlines), "Should classify all headlines"
# INTERPRETATION: Every prediction above is a news topic, so none of them
# routes a filing to a regulation team. The token-importance list shows
# where the trained first layer's attention went; treat it as a debugging
# view, not as an explanation a regulator could audit (attention weights
# are not guaranteed to reflect what drove the output).
print("\n--- Checkpoint 5 passed --- routing check complete\n")



## REFLECTION


[x] Built multi-head attention wrapping the from-scratch attention kernel
  [x] Explained how different heads capture different relationship types
  [x] Implemented sinusoidal positional encoding (word order for transformers)
  [x] Built a full TransformerClassifier with nn.TransformerEncoder
  [x] Trained on full AG News (120K headlines), test acc: {transformer_test_acc:.1%}
  [x] Visualised per-head attention patterns
  [x] Checked a regulatory-routing use case against the model's label space

  KEY INSIGHT:
    Multi-head attention is like having multiple specialists read the same
    document simultaneously. One head notices syntax, another notices
    entities, another notices sentiment. Together they capture a richer
    understanding than any single attention computation could.

  Next: In 03_lstm_baseline.py, you'll build an LSTM baseline to see
  exactly what the Transformer's attention mechanism buys us compared
  to sequential processing.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Transformer Encoder")
print("=" * 70)
print(
)

